# 04 — Calibration gán nhãn và đo độ đồng thuận

Trước khi gán nhãn hàng nghìn review, tôi cần biết guideline 4 nhãn (BUG / FEATURE / UX / OTHER) đã đủ rõ để hai người đọc
cùng một review thì ra cùng một nhãn hay chưa. Nếu chưa, mọi chỉ số F1 về sau đều thiếu cơ sở, vì nhãn tham chiếu không đáng tin.
Vì vậy tôi làm một vòng calibration với hai người gán độc lập rồi đo độ đồng thuận (Q10).

Cách tôi tổ chức:

- Lấy ngẫu nhiên 150 review Shopee (seed 42) từ corpus trung gian và chia thành hai batch, mỗi batch 75 review, đúng như đề cương
  §7.2.1. Batch 1 dùng để hai người gán rồi cùng ngồi xem những chỗ lệch nhau và sửa guideline; batch 2 gán bằng guideline đã sửa và
  là batch tôi dùng để quyết định guideline đã đạt hay chưa.
- Bảng gán chỉ có `review_id` và nội dung review (bản gốc đã ẩn danh hóa, giữ nguyên emoji và viết tắt). Tôi cố ý ẩn số sao, ngày và
  phiên bản app để người gán không bị số sao dẫn dắt (Q17).
- Ngoài cột nhãn còn có cột `ambiguous` (đánh 1 khi review có hai ý ngang nhau, khó chọn ý chính) và cột `note` để ghi lý do khi
  phân vân — phần ghi chú này rất có ích khi họp sửa guideline.
- Hai người làm trên hai bản sao Google Sheets riêng và không xem bản của nhau.
- Ngưỡng tôi đặt là Cohen's κ ≥ 0.6 trên batch 2. Tuy vậy κ chỉ là điều kiện cần; tôi sẽ xem thêm cụ thể các ca lệch nhau giữa
  FEATURE và UX, và giữa nhóm actionable với OTHER (tương ứng tầng 1 của mô hình), vì đây là hai chỗ dễ nhầm nhất.

Sau batch 1 (κ = 0,671, 9 ca lệch), hai người đã họp và tôi cập nhật guideline lên v1.1 với 5 quy tắc phân xử R1–R5 và quy tắc đánh `ambiguous` (xem `00_protocol/edge_cases_log.md`); batch 2 được gán bằng guideline v1.1.

150 review calibration được ghi vào manifest để loại khỏi Hold-out sau này; sau khi thống nhất nhãn, chúng sẽ được đưa vào Train
(Q9). Phần xử lý nằm trong `src/annotation/calibration.py` và `src/annotation/agreement.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import load_jsonl
from src.annotation.calibration import (draw_calibration, export_blank, write_manifest, INSTRUCTIONS,
                                        FEATURE_UX_KEYWORDS, draw_keyword_set, write_keyword_manifest)
from src.annotation.agreement import (merge, agreement, disagreements_by_pair, gate_checklist,
                                      build_finalize_rows, write_finalize)
set_seed(42)
TAG = 'T0_2026-10-06'
SOURCE = os.path.join(input_dir('interim'), f'shopee_clean_{TAG}.jsonl')
CAL_DIR = os.path.abspath(os.path.join('..', '07_human_eval', 'calibration'))
MANIFEST = os.path.join(input_dir('manifests'), f'calibration_manifest_{TAG}.json')

## Rút mẫu và xuất bảng gán nhãn

In [2]:
records = load_jsonl(SOURCE)
batches = draw_calibration(records, n=150, n_batches=2, seed=42)
sheets = {b: os.path.join(CAL_DIR, f'calibration_{b}_blank.csv') for b in batches}
if all(os.path.exists(p) for p in sheets.values()) and os.path.exists(MANIFEST):
    print('Bảng gán nhãn và manifest đã tồn tại — giữ nguyên (không ghi đè).')
else:
    for b, rows in batches.items():
        export_blank(rows, sheets[b])
    write_manifest(batches, SOURCE, sheets, MANIFEST, seed=42)
with open(MANIFEST, encoding='utf-8') as f:
    manifest = json.load(f)
for b, info in manifest['batches'].items():
    print(f"{b}: {len(info['review_ids'])} review → {info['sheet']}")
print('Manifest:', MANIFEST)
print()
print(INSTRUCTIONS)

Bảng gán nhãn và manifest đã tồn tại — giữ nguyên (không ghi đè).
batch1: 75 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_batch1_blank.csv
batch2: 75 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_batch2_blank.csv
Manifest: C:\Workspace\kltn-vn-app-review-requirements\01_data\manifests\calibration_manifest_T0_2026-10-06.json

Hướng dẫn cho người gán nhãn
- Nhãn hợp lệ ở cột `label`: BUG / FEATURE / UX / OTHER (theo `00_protocol/annotation_guideline_v1.md` và `decision_tree.md`).
- Cột `ambiguous`: điền 1 nếu review có hai ý actionable độc lập ngang nhau, không xác định được ý chính; để trống nếu không.
  Vẫn chọn MỘT nhãn tốt nhất ở cột `label`.
- Cột `note`: ghi ngắn lý do khi phân vân.
- Mỗi người làm trên BẢN SAO Google Sheets riêng (A và B), gán độc lập, không xem bản của người kia, không thảo luận khi đang gán.
- Cột `label` dùng Data validation dạng dropdown với 4 giá trị trên; cột `ambigu

In [3]:
# Xem nhanh vài dòng của batch 1 (đúng như người gán sẽ thấy)
import pandas as pd
pd.read_csv(sheets['batch1'], encoding='utf-8-sig').head(8)

,review_id,text,label,ambiguous,note
0,75db5c2c-46f8-4339-afaf-2f5ae33ba3f5,tốt,NaN,NaN,NaN
1,93d24e85-3896-4e63-97db-d8be6af3967f,trả lại cái lựa chọn đơn vị vận chuyển đi tại ...,NaN,NaN,NaN
2,3966135b-6b5c-4722-a125-469f65a18d40,tìm kiếm trên shoppee bị cái gì mà ko ra v cứ ...,NaN,NaN,NaN
3,45484b4a-c5eb-4bda-8547-7d2185b247de,ngon,NaN,NaN,NaN
4,b6d4b63e-9169-4f1c-935f-1ad4cb2cca10,cho voucher rồi lúc mua thì bảo là voucher ảo ...,NaN,NaN,NaN
5,4958811e-744a-4a0c-aa06-edc68b6fc1bc,chính sách trả hàng cực kỳ tệ. người bán gửi s...,NaN,NaN,NaN
6,fb0c4bbc-4705-402a-b569-25afa2dc25c7,"mỗi lần vào là vào thẳng Live & video , tôi kh...",NaN,NaN,NaN
7,67b0d008-530c-4681-8001-a253c98f4548,nó cứ bị dính m02 đồ,NaN,NaN,NaN


## Đo độ đồng thuận

Phần này chỉ có kết quả sau khi hai người gán xong và tôi đã tải về `calibration_<batch>_A.csv`, `calibration_<batch>_B.csv` vào
`07_human_eval/calibration/`; trước đó notebook chỉ báo đang chờ. Kết quả batch 1 dùng để thảo luận, còn batch 2 dùng để quyết định.

In [4]:
GATE_BATCH = 'batch2'
results = {}
for b in manifest['batches']:
    pa, pb = (os.path.join(CAL_DIR, f'calibration_{b}_{w}.csv') for w in ('A', 'B'))
    if not (os.path.exists(pa) and os.path.exists(pb)):
        print(f'{b}: đang chờ {os.path.basename(pa)} và {os.path.basename(pb)}')
        continue
    m = merge(pa, pb, expected_ids=manifest['batches'][b]['review_ids'])
    if m['problems']:
        print(f'{b}: cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
        continue
    results[b] = {'df': m['df'], 'res': agreement(m['df'])}
    results[b]['groups'] = disagreements_by_pair(m['df'])
    r = results[b]['res']
    print(f"===== {b}: n={r['n_valid']} | % đồng thuận={r['percent_agreement']} | κ 4 lớp={r['kappa_4class']}")
    print(f"  actionable–OTHER: % đồng thuận={r['actionable_other']['percent_agreement']} | κ={r['actionable_other']['kappa']}")
    print(f"  FEATURE–UX (cả hai chọn FEATURE/UX): n={r['feature_ux']['n_both_feature_or_ux']} | "
          f"% đồng thuận={r['feature_ux']['percent_agreement']} | κ={r['feature_ux']['kappa']}")
    print('  Ma trận nhầm lẫn (hàng = A, cột = B):')
    print(r['confusion'].to_string())

===== batch1: n=75 | % đồng thuận=88.0 | κ 4 lớp=0.671
  actionable–OTHER: % đồng thuận=90.7 | κ=0.728
  FEATURE–UX (cả hai chọn FEATURE/UX): n=2 | % đồng thuận=0.0 | κ=0.0
  Ma trận nhầm lẫn (hàng = A, cột = B):
B        BUG  FEATURE  UX  OTHER
A                               
BUG       11        0   0      0
FEATURE    0        0   2      1
UX         0        0   0      2
OTHER      2        2   0     55
===== batch2: n=75 | % đồng thuận=89.3 | κ 4 lớp=0.785
  actionable–OTHER: % đồng thuận=93.3 | κ=0.849
  FEATURE–UX (cả hai chọn FEATURE/UX): n=6 | % đồng thuận=100.0 | κ=1.0
  Ma trận nhầm lẫn (hàng = A, cột = B):
B        BUG  FEATURE  UX  OTHER
A                               
BUG       13        0   0      0
FEATURE    2        1   0      0
UX         1        0   5      0
OTHER      4        1   0     48


### Các ca hai người gán khác nhau

Tôi nhóm các ca lệch theo cặp nhãn (ví dụ A chọn FEATURE, B chọn UX), kèm nội dung review và ghi chú của cả hai người, để mang vào
buổi họp sửa guideline.

In [5]:
for b, out in results.items():
    print(f'===== {b}')
    for (la, lb), rows in out['groups'].items():
        print(f'--- A={la} / B={lb}: {len(rows)} ca')
        for r in rows:
            print(f"  [{r['review_id'][:8]}] {r['text'][:200]}")
            if r['note_a'] or r['note_b']:
                print(f"      note A: {r['note_a'] or '-'} | note B: {r['note_b'] or '-'}")
if not results:
    print('Chưa có batch nào đủ hai tệp gán nhãn.')

===== batch1
--- A=OTHER / B=FEATURE: 2 ca
  [29248bd9] mong thêm nhiều giảm giá
      note A: Yêu cầu thêm giảm giá, thuộc khuyến mãi/chính sách | note B: Đề xuất thêm hoặc mở rộng giảm giá, là feature request
  [9e89d23d] bỏ thêm tủ đồ đi nhé, không đặt hàng dc
      note A: Yêu cầu thêm sản phẩm/tủ đồ, không phải yêu cầu sửa chức năng app | note B: Đề xuất bỏ hoặc thêm tủ đồ để đặt hàng được, là feature request
--- A=OTHER / B=BUG: 2 ca
  [42a8462c] mk vừa lập acc lưu đc viu chờ 80k xg e nó ghi là 3/8 hết hạn r mk để đấy chx đc 3 ngày viu chờ ko còn nó lại cho mk viu chờ 60k thôi bực ghê
      note A: Voucher/ưu đãi hết hạn hoặc thay đổi, thuộc chính sách khuyến mãi; chưa đủ căn cứ là lỗi app | note B: Voucher bị mất hoặc hết hạn sai, là lỗi voucher
  [8e0c8551] đang mua thì bị hết pin, bị thu dth, mong nhà phát hành sửa lỗi
      note A: Nêu tình huống thiết bị/không đủ thông tin về lỗi app cụ thể | note B: Lỗi trong quá trình mua như hết pin hoặc bị thu điện thoại, là malfunction


## Có chốt được guideline chưa

Bảng tổng hợp dưới đây gồm κ tổng, κ ở hai chỗ dễ nhầm, số ca lệch theo từng nhóm và số ca được đánh dấu ambiguous. Nếu κ của
batch 2 dưới 0.6, hoặc FEATURE và UX vẫn lệch nhau nhiều, tôi sẽ chưa chốt guideline mà dùng chính các ca lệch này để họp sửa, rồi
làm thêm một batch calibration nữa. Nếu đạt, bước chốt guideline (thêm các ví dụ thật từ calibration vào
`annotation_guideline_v1.md`) sẽ được làm riêng sau khi tôi xem lại kết quả.

In [6]:
for b, out in results.items():
    print(gate_checklist(out['res'], out['groups'], b, is_gate_batch=(b == GATE_BATCH)))
    print()
if not results:
    print('Chưa đủ dữ liệu để lập checklist.')

Checklist chốt guideline — batch1
  κ tổng (4 lớp)          : 0.671  (cổng ≥ 0.6)
  % đồng thuận            : 88.0%  (n=75)
  κ actionable–OTHER      : 0.728  | bất đồng: 7
  κ FEATURE–UX (tập con)  : 0.0  | n tập con: 2 | bất đồng FEATURE↔UX: 2
  Đánh dấu ambiguous      : A=1, B=1
  Số ca bất đồng theo cặp (A/B): OTHER/FEATURE=2, OTHER/BUG=2, UX/OTHER=2, FEATURE/UX=2, FEATURE/OTHER=1
  → Batch thảo luận: dùng các ca bất đồng ở trên để họp sửa guideline trước khi gán batch tiếp theo.

Checklist chốt guideline — batch2
  κ tổng (4 lớp)          : 0.785  (cổng ≥ 0.6)
  % đồng thuận            : 89.3%  (n=75)
  κ actionable–OTHER      : 0.849  | bất đồng: 5
  κ FEATURE–UX (tập con)  : 1.0  | n tập con: 6 | bất đồng FEATURE↔UX: 0
  Đánh dấu ambiguous      : A=1, B=1
  Số ca bất đồng theo cặp (A/B): OTHER/BUG=4, FEATURE/BUG=2, UX/BUG=1, OTHER/FEATURE=1
  → κ đạt cổng. Người nghiên cứu xem thêm 0 ca bất đồng FEATURE/UX trước khi kết luận; nếu không còn vấn đề hệ thống: guideline sẵn sàng chố

## Bộ bổ sung để kiểm tra ranh giới FEATURE–UX

Batch 1 lấy ngẫu nhiên nên phần lớn là OTHER: chỉ khoảng 3–5 review được gán FEATURE hoặc UX, và κ ở ranh giới FEATURE–UX chỉ
tính được trên 2 ca, gần như không có ý nghĩa. Batch 2 nhiều khả năng cũng vậy, nên cổng κ không cho tôi biết hai người có hiểu
giống nhau ở chỗ này hay không — trong khi đây lại đúng là chỗ vừa thêm quy tắc R3.

Vì vậy tôi rút thêm khoảng 30 review có chứa các cụm từ hay gặp trong góp ý tính năng hoặc phàn nàn trải nghiệm ("nên thêm",
"mong có", "bổ sung", "khó dùng", "khó tìm", "không tìm thấy", "rối", "phiền", "cho tắt", "tùy chọn"), loại những review đã nằm
trong batch 1 và batch 2. Lần rút đầu tiên tôi lấy ngẫu nhiên trên toàn bộ review khớp keyword thì 27/30 review chứa chữ "phiền"
(cụm này chiếm khoảng 80% số review khớp, phần lớn là phàn nàn quảng cáo đã được R2 xử lý), nên tôi đổi sang rút đều 3 review
cho mỗi nhóm keyword. Hai người gán thử độc lập bằng guideline v1.1 rồi cùng xem lại. Bộ này chỉ dùng để thảo luận ranh giới,
**không** tính vào κ cổng; cổng vẫn đo trên batch 2 lấy ngẫu nhiên (amendment v1.6). Vì người gán đã đọc các review này, chúng
cũng được ghi manifest riêng để loại khỏi Hold-out sau này.

In [7]:
KW_SHEET = os.path.join(CAL_DIR, 'calibration_feature_ux_blank.csv')
KW_MANIFEST = os.path.join(input_dir('manifests'), f'calibration_feature_ux_manifest_{TAG}.json')
used_ids = [rid for info in manifest['batches'].values() for rid in info['review_ids']]
if os.path.exists(KW_SHEET) and os.path.exists(KW_MANIFEST):
    print('Bộ bổ sung FEATURE–UX đã tồn tại — giữ nguyên (không ghi đè).')
else:
    kw_rows, n_cand, hits = draw_keyword_set(records, n=30, exclude_ids=used_ids, seed=42)
    export_blank(kw_rows, KW_SHEET)
    write_keyword_manifest(kw_rows, FEATURE_UX_KEYWORDS, n_cand, SOURCE, KW_SHEET, KW_MANIFEST, seed=42)
    print(f'Số review khớp keyword (sau khi loại batch 1/2): {n_cand:,}')
    print('Số review rút theo từng nhóm keyword (3 mỗi nhóm):', hits)
with open(KW_MANIFEST, encoding='utf-8') as f:
    kw_manifest = json.load(f)
print(f"Bộ bổ sung: {len(kw_manifest['review_ids'])} review → {KW_SHEET}")
print('Trùng với batch 1/2:', len(set(kw_manifest['review_ids']) & set(used_ids)))

Bộ bổ sung FEATURE–UX đã tồn tại — giữ nguyên (không ghi đè).
Bộ bổ sung: 30 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_feature_ux_blank.csv
Trùng với batch 1/2: 0


### Đo trên bộ bổ sung (không phải κ cổng)

Phần này chạy khi đã có `calibration_feature_ux_A.csv` và `calibration_feature_ux_B.csv`. Kết quả chỉ để xem hai người có tách
FEATURE, UX và BUG giống nhau hay không; nó không được dùng để quyết định chốt guideline.

In [8]:
pa, pb = (os.path.join(CAL_DIR, f'calibration_feature_ux_{w}.csv') for w in ('A', 'B'))
if not (os.path.exists(pa) and os.path.exists(pb)):
    print('Bộ bổ sung FEATURE–UX: đang chờ calibration_feature_ux_A.csv và calibration_feature_ux_B.csv')
else:
    m = merge(pa, pb, expected_ids=kw_manifest['review_ids'])
    if m['problems']:
        print('Cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
    else:
        kw_res = agreement(m['df'])
        print('KHÔNG phải κ cổng — chỉ để thảo luận ranh giới FEATURE–UX.')
        print(f"n={kw_res['n_valid']} | % đồng thuận={kw_res['percent_agreement']} | κ 4 lớp={kw_res['kappa_4class']}")
        print(f"FEATURE–UX (cả hai chọn FEATURE/UX): n={kw_res['feature_ux']['n_both_feature_or_ux']} | "
              f"% đồng thuận={kw_res['feature_ux']['percent_agreement']} | κ={kw_res['feature_ux']['kappa']}")
        print('Ma trận nhầm lẫn FEATURE/UX/BUG (hàng = A, cột = B):')
        print(kw_res['confusion'].loc[['FEATURE', 'UX', 'BUG'], ['FEATURE', 'UX', 'BUG']].to_string())
        for (la, lb), rows in disagreements_by_pair(m['df']).items():
            print(f'--- A={la} / B={lb}: {len(rows)} ca')
            for r in rows:
                print(f"  [{r['review_id'][:8]}] {r['text'][:200]}")
                if r['note_a'] or r['note_b']:
                    print(f"      note A: {r['note_a'] or '-'} | note B: {r['note_b'] or '-'}")

KHÔNG phải κ cổng — chỉ để thảo luận ranh giới FEATURE–UX.
n=30 | % đồng thuận=86.7 | κ 4 lớp=0.814
FEATURE–UX (cả hai chọn FEATURE/UX): n=14 | % đồng thuận=92.9 | κ=0.851
Ma trận nhầm lẫn FEATURE/UX/BUG (hàng = A, cột = B):
B        FEATURE  UX  BUG
A                        
FEATURE        8   1    1
UX             0   5    1
BUG            0   1    2
--- A=FEATURE / B=BUG: 1 ca
  [7447906e] không cài face id mà thanh toán bắt dùng face id. cũng không cho tắt😡
      note A: Thanh toán bắt Face ID dù người dùng chưa cài và không có tùy chọn tắt; yêu cầu thêm lựa chọn xác thực là FEATURE. | note B: bat buoc dung face id du chua cai, khong cho tat, malfunction
--- A=FEATURE / B=UX: 1 ca
  [7825c8f5] bắt ép coi live và video khi mở app, lướt mua hàng tới đâu cũng hiện video gây khó chịu. không cho tắt và không có cách liên hệ hỗ trợ. cực kỳ phiền
      note A: Hành vi tự mở Live hoặc video kèm yêu cầu không cho tắt; theo R3, đề xuất thêm tùy chọn thay đổi là FEATURE. | note B: hanh vi da 

## Chốt guideline và chốt nhãn calibration

Batch 2 đạt cổng (κ = 0,785) và hai người không lệch nhau ở FEATURE/UX, nhưng 7/8 ca lệch lại dồn quanh BUG: một người hay gán
"malfunction" cho những thao tác app không hỗ trợ, hoặc cho kết quả do chính sách và giao hàng. Chúng tôi thống nhất thêm quy tắc
R6 ("app làm sai điều lẽ ra làm được" là BUG, "app không làm được điều tôi muốn" là FEATURE, kết quả do chính sách/chống gian
lận/logistics là OTHER) và chốt luôn guideline ở bản v1.2 mà không gán thêm batch. Lý do là κ đã vượt cổng khá xa và các ca lệch
chỉ nằm ở một ranh giới; Hold-out sẽ được hai người gán toàn bộ nên R6 sẽ được kiểm lại bằng chính độ đồng thuận ở đó.

Vì 180 review calibration sẽ được đưa vào Train (Q9), nhãn của chúng phải khớp guideline đã đóng băng. Khi chọn ví dụ cho guideline,
tôi nhận ra một số review hai người cùng gán giống nhau nhưng lại trái quy tắc mới — ví dụ quảng cáo bị gán BUG ở batch 1 (trước
khi có R2), hay đơn bị hủy vì "hoạt động bất thường" bị gán BUG trong khi R6 coi đây là quyết định của hệ thống chống gian lận.
Vì vậy tôi rà lại toàn bộ 180 review và xuất một file gồm tất cả ca lệch cùng các ca đồng thuận nhưng trái v1.2
(`agreed_conflicts_v1.2`). Cột `R_rule` ghi quy tắc liên quan và nhãn đã chốt (nếu có) hoặc nhãn gợi ý; cột `agreed_label` để
trống để hai người điền nhãn cuối.

In [9]:
CAL_SETS = {s: (os.path.join(CAL_DIR, f'calibration_{s}_A.csv'), os.path.join(CAL_DIR, f'calibration_{s}_B.csv'))
            for s in ('batch1', 'batch2', 'feature_ux')}

# Ghi chú theo quy tắc cho từng ca lệch (mã = 8 ký tự đầu review_id). "đã chốt" = đã thống nhất, ghi trong edge_cases_log.md
RULES = {
    # batch 1 (đã chốt, guideline v1.1)
    '29248bd9': 'R1 → OTHER (đã chốt)', '42a8462c': 'R1 → OTHER (đã chốt)',
    '42acd7bc': 'R2 → UX (đã chốt)', '79bd9683': 'R2 → UX (đã chốt)',
    'dad71cb2': 'R3 → FEATURE (đã chốt)', 'fb0c4bbc': 'R3 + quy tắc ambiguous → FEATURE, ambiguous=1 (đã chốt)',
    '93d24e85': 'R4 → FEATURE (đã chốt)', '8e0c8551': 'R5 → OTHER (đã chốt)', '9e89d23d': 'R5 → OTHER (đã chốt)',
    # batch 2 (đã chốt, guideline v1.2)
    '92402354': 'R6 → FEATURE (đã chốt)', 'e63ed42d': 'R6 → FEATURE (đã chốt)', 'aa4bf2af': 'R6 → OTHER (đã chốt)',
    '99aa591f': 'R6 → OTHER (đã chốt)', '75be2c08': 'R6 → OTHER (đã chốt)', '481654e7': 'R5 → OTHER (đã chốt)',
    # bộ bổ sung (đã chốt, guideline v1.2)
    '7447906e': 'R6 + R3 bổ sung → FEATURE (đã chốt)', '7825c8f5': 'R3 bổ sung → FEATURE (đã chốt)',
    # CHƯA chốt — chỉ gợi ý quy tắc để hai người thảo luận
    '63848f1a': 'CHƯA CHỐT — quy tắc ambiguous: nhiều ý khác lớp; hai người khác nhau về lớp của ý đầu '
                '(tin nhắn lừa đảo vẫn đến: UX theo R2 hay BUG?)',
    'cf7471aa': 'CHƯA CHỐT — R4 (yêu cầu đổi phương thức hoàn tiền → FEATURE) hay R6 (chính sách hoàn tiền → OTHER)?',
    '9d8a3972': 'CHƯA CHỐT — quy tắc ambiguous: không tìm thấy mục sửa địa chỉ (UX) + "mã giảm lỗi tùm lum" (BUG hay R5 mơ hồ?)',
    'c6662107': 'CHƯA CHỐT — "đừng làm rối địa chỉ … trả lại đơn": R5 mơ hồ (OTHER), BUG (xử lý địa chỉ sai) hay UX?',
}

# Ca hai người ĐỒNG THUẬN nhưng nhãn trái guideline v1.2 — nhãn bên phải chỉ là GỢI Ý
AGREED_CONFLICTS = {
    # R6 (làm rõ 12/10): hệ thống chống gian lận / chính sách → OTHER
    'f8fce12e': 'R6 — đơn tự hủy khi áp voucher (chống gian lận) → gợi ý OTHER',
    '93c18679': 'R6 — tự động hủy đơn + không cho trả hàng (chính sách) → gợi ý OTHER',
    'b6d4b63e': 'R6 — "voucher ảo" rồi khóa sđt (chống gian lận) → gợi ý OTHER',
    'da1855df': 'R6 — báo lỗi rồi khóa tài khoản (chính sách) → gợi ý OTHER',
    '62690671': 'R6 — không nhận xu vì "hoạt động bất thường" → gợi ý OTHER',
    'c42ebf79': 'R6 — "giao dịch bất thường", không đặt được → gợi ý OTHER',
    'd076ab22': 'R6 — tự hủy đơn vì "bất thường" → gợi ý OTHER',
    '296f3681': 'R6 — đơn toàn tự bị hủy (chống gian lận) → gợi ý OTHER',
    '0f8c12ea': 'R6 — đặt gì cũng hủy "kêu bất thường" → gợi ý OTHER',
    '29f95921': 'R6 — có voucher xong đặt thì hủy → gợi ý OTHER',
    '682e8322': 'R6 — "ko cho trả hàng" (chính sách); batch 1 có 87de4734 "ko cho hoàn hàng" đã đồng thuận OTHER → gợi ý OTHER',
    # R2: quảng cáo / tự chuyển sang app → UX
    '8466ce83': 'R2 — đang xem phim cứ tự nhảy sang Shopee (quảng cáo chuyển app) → gợi ý UX',
    'd093c4f1': 'R2 — quảng cáo hiện liên tục, tự chuyển sang app → gợi ý UX',
    # ranh giới — cần hai người cân nhắc
    'cce5f87f': 'RANH GIỚI — "k mua được": R5 (mơ hồ → OTHER) hay BUG (không mua được, như "không đăng nhập được" của Q16)?',
    '67b0d008': 'RANH GIỚI — "dính m02": mã lỗi kỹ thuật (BUG) hay mã hạn chế tài khoản (R6 → OTHER)?',
    '2161fc14': 'RANH GIỚI — SPayLater hoàn vào số dư Shopee không rút được: chính sách (R6 → OTHER) hay BUG?',
}

FINALIZE = os.path.join(CAL_DIR, 'disagreements_to_finalize.csv')
fin_rows = build_finalize_rows(CAL_SETS, RULES, AGREED_CONFLICTS)
if os.path.exists(FINALIZE):
    print('File chốt nhãn đã tồn tại — giữ nguyên (có thể đã điền agreed_label).')
else:
    write_finalize(fin_rows, FINALIZE)
    print('Đã xuất:', FINALIZE)
fin = pd.DataFrame(fin_rows)
print(fin['source'].value_counts().to_string())
print('Chưa chốt (cần thảo luận):', int(fin['R_rule'].str.startswith('CHƯA CHỐT').sum()),
      '| ranh giới:', int(fin['R_rule'].str.startswith('RANH GIỚI').sum()))

File chốt nhãn đã tồn tại — giữ nguyên (có thể đã điền agreed_label).


source
agreed_conflicts_v1.2      16
batch1_disagreement         9
batch2_disagreement         8
feature_ux_disagreement     4
Chưa chốt (cần thảo luận): 4 | ranh giới: 3


## Ghi nhãn cuối của 180 review calibration vào gold

Sau khi hai người điền xong `agreed_label` và `agreed_ambiguous` cho 37 ca trong file chốt nhãn, tôi dựng nhãn cuối cho toàn bộ
180 review calibration (batch 1, batch 2 và bộ bổ sung): review nào có trong file chốt nhãn thì lấy nhãn đã thống nhất (kể cả các ca
trước đó hai người gán giống nhau nhưng trái guideline v1.2); các review còn lại là những ca hai người gán giống nhau và khớp v1.2,
nên lấy luôn nhãn đồng thuận. Với các ca đồng thuận, tôi chỉ ghi `ambiguous_flag = 1` khi cả hai người cùng đánh dấu.

Kết quả được ghi thành hai file. File `gold/adjudicated/calibration_gold_v1.2.csv` là file sẽ dùng để train, chỉ có văn bản và nhãn,
không có số sao hay ngày (Q17, và đề cương dùng văn bản làm đầu vào duy nhất). File `gold/provenance.csv` lưu đầy đủ truy vết theo
Phụ lục B của đề cương: nhãn của từng người, nhãn cuối, cách ra nhãn (đồng thuận hay hội ý), phiên bản guideline, cùng số sao và
ngày review làm metadata cho phân tích thời gian sau này. Cả 180 review được ghi vào danh sách loại khỏi Hold-out (Q9).

In [10]:
from datetime import date
from src.annotation.gold import (load_finalized, build_calibration_gold, write_provenance, write_train_gold,
                                 write_holdout_exclusion, PROVENANCE_COLUMNS)
GOLD_DIR = input_dir('gold')
finalized = load_finalized(FINALIZE)
print(f'File chốt nhãn: {len(finalized)} ca, nhãn hợp lệ, không ô trống')
print('  agreed_label:', pd.Series([v[0] for v in finalized.values()]).value_counts().to_dict(),
      '| agreed_ambiguous = 1:', sum(v[1] for v in finalized.values()))

interim_by_id = {r['review_id']: r for r in records}
gold_rows, gstats = build_calibration_gold(CAL_SETS, finalized, interim_by_id, guideline_version='v1.2',
                                           label_date=date.today().isoformat())
g = pd.DataFrame(gold_rows)
print(f"\nNhãn cuối: {len(g)} review | từ file chốt nhãn: {gstats['from_finalize']} | đồng thuận khớp v1.2: {gstats['consensus']}")
print('Theo bộ:', g['calibration_set'].value_counts().to_dict())
print('Phân bố nhãn cuối:', g['gold_label'].value_counts().reindex(['BUG', 'FEATURE', 'UX', 'OTHER']).to_dict(),
      '| ambiguous_flag = 1:', int(g['ambiguous_flag'].sum()))
print(pd.crosstab(g['calibration_set'], g['gold_label']).reindex(columns=['BUG', 'FEATURE', 'UX', 'OTHER'], fill_value=0).to_string())
if gstats['ambiguous_one_sided']:
    print('Ca đồng thuận nhãn nhưng chỉ MỘT người đánh ambiguous (ghi ambiguous_flag = 0):',
          [x[:8] for x in gstats['ambiguous_one_sided']])

File chốt nhãn: 37 ca, nhãn hợp lệ, không ô trống
  agreed_label: {'OTHER': 23, 'FEATURE': 8, 'UX': 6} | agreed_ambiguous = 1: 3

Nhãn cuối: 180 review | từ file chốt nhãn: 37 | đồng thuận khớp v1.2: 143
Theo bộ: {'batch1': 75, 'batch2': 75, 'feature_ux': 30}
Phân bố nhãn cuối: {'BUG': 10, 'FEATURE': 17, 'UX': 16, 'OTHER': 137} | ambiguous_flag = 1: 6


gold_label       BUG  FEATURE  UX  OTHER
calibration_set                         
batch1             2        3   4     66
batch2             6        4   6     59
feature_ux         2       10   6     12


In [11]:
PROVENANCE = os.path.join(GOLD_DIR, 'provenance.csv')
TRAIN_GOLD = os.path.join(GOLD_DIR, 'adjudicated', 'calibration_gold_v1.2.csv')
EXCLUSION = os.path.join(input_dir('manifests'), f'holdout_exclusion_{TAG}.json')
if os.path.exists(TRAIN_GOLD):
    print('Gold calibration đã được ghi trước đó — giữ nguyên.')
else:
    write_train_gold(gold_rows, TRAIN_GOLD)
    write_provenance(gold_rows, PROVENANCE)
    write_holdout_exclusion(gold_rows, EXCLUSION)
prov = pd.read_csv(PROVENANCE, encoding='utf-8-sig')
excl = json.load(open(EXCLUSION, encoding='utf-8'))
print('provenance.csv:', len(prov), 'dòng | đủ cột theo Phụ lục B + truy vết:', list(prov.columns) == PROVENANCE_COLUMNS,
      '| có ambiguous_flag:', 'ambiguous_flag' in prov.columns)
print('Cột:', list(prov.columns))
print('Train gold:', TRAIN_GOLD, '| cột:', list(pd.read_csv(TRAIN_GOLD, encoding='utf-8-sig').columns))
print(f"Loại khỏi Hold-out: {excl['n']} review_id — {excl['by_set']}")

provenance.csv: 180 dòng | đủ cột theo Phụ lục B + truy vết: True | có ambiguous_flag: True
Cột: ['review_id', 'app', 'content', 'raw_text_hash', 'rating', 'review_date', 'app_version', 'sampling_source', 'calibration_set', 'split', 'label_a', 'label_b', 'agreement_flag', 'gold_label', 'ambiguous_flag', 'guideline_version', 'annotated_with_guideline', 'labeled_by', 'resolution', 'label_date']


Train gold: C:\Workspace\kltn-vn-app-review-requirements\01_data\gold\adjudicated\calibration_gold_v1.2.csv | cột: ['review_id', 'app', 'text', 'gold_label', 'ambiguous_flag', 'sampling_source', 'split', 'guideline_version']
Loại khỏi Hold-out: 180 review_id — {'batch1': 75, 'batch2': 75, 'feature_ux': 30}
